# New profile-analysis functions in PyLake

Every function below has an explanation, a minimal executable demonstration, and an interpretation. Public analysis functions come first; internal helpers are shown later so contributors can understand the implementation.

In [ ]:
import numpy as np
import xarray as xr
import pylake
from pylake import functions as fn

depth = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
temperature = np.array([14.3, 14, 12.1, 10, 9.7, 9.5, 6, 5], dtype=float)
bthD = np.array([0, 2.3, 2.5, 4.2, 5.8, 8], dtype=float)
bthA = np.array([100, 90, 86, 82, 20, 0], dtype=float)

## 1. `depth_filter`

CTDs often record a soak near the surface, small upward movements, and a final upcast. `depth_filter` retains the monotonic downcast. With `index=True`, it returns original row indices so all other sensor columns can be filtered identically.

In [ ]:
raw_depth = np.array([0, 0.1, 0.05, 0.2, 0.8, 1.5, 1.4, 2.0, 3.0, 2.5])
filtered_depth = fn.depth_filter(raw_depth, run_length=2)
kept_rows = fn.depth_filter(raw_depth, run_length=2, index=True)
filtered_depth, kept_rows

## 2. `depth_average`

Repeated depths are collapsed and their finite measurements are averaged. This is useful before interpolation and stratification calculations.

In [ ]:
fn.depth_average([1, 1, 2, 2, 3], [20, 22, 18, 20, 15])

## 3. `thermocline` and `weighted`

The maximum density-gradient interval identifies the thermocline. `weighted=True` (default) refines the position between sensors using Read et al. (2011). `weighted=False` returns the interval midpoint and is useful for debugging and direct interval comparisons.

In [ ]:
weighted_depth, weighted_index = pylake.thermocline(temperature, depth)
midpoint_depth, midpoint_index = pylake.thermocline(temperature, depth, weighted=False)
weighted_depth, weighted_index, midpoint_depth, midpoint_index

## 4. `seasonal_thermocline`

This selects the deepest density-gradient peak above `Smin`, falling back to the diurnal thermocline when no peak qualifies. `seasonal_smoothed=False` avoids time-series smoothing for a single profile.

In [ ]:
pylake.seasonal_thermocline(
    temperature,
    depth,
    seasonal_smoothed=False,
    weighted=True,
)

## 5. `center_buoyancy`

This is the depth-weighted center of positive buoyancy frequency. A stratified profile returns a depth; a uniform profile returns `NaN`.

In [ ]:
fn.center_buoyancy(temperature, depth), fn.center_buoyancy(np.full(8, 10.0), depth)

## 6. Layer averages

`layer_average` interpolates both observations and lake area on a fine vertical grid and calculates a volume-weighted mean. `layer_temperature` applies it directly to temperature; `layer_density` first converts temperature and salinity to density.

In [ ]:
layer_value = fn.layer_average(0, 4, temperature, depth, bthA, bthD)
layer_temp = fn.layer_temperature(0, 4, temperature, depth, bthA, bthD)
layer_rho = fn.layer_density(0, 4, temperature, depth, bthA, bthD, sal=0.2)
layer_value, layer_temp, layer_rho

## 7. Whole-lake, epilimnion, and hypolimnion temperature

These wrappers make layer boundaries explicit. `meta_top` is the top of the metalimnion and `meta_bottom` is its bottom.

In [ ]:
whole = fn.whole_lake_temperature(temperature, depth, bthA, bthD)
epi = fn.epi_temperature(temperature, depth, bthA, bthD, meta_top=2.5)
hypo = fn.hypo_temperature(temperature, depth, bthA, bthD, meta_bottom=5)
whole, epi, hypo

## 8. `ustar`

`ustar` converts wind speed into water friction velocity. Inputs are wind speed in m/s, measurement height in m, and mean epilimnion density in kg/m³.

In [ ]:
fn.ustar(wind_speed=[2, 5, 10], wind_height=10, average_epi_density=998)

## 9. Input helpers: `control`, `format_Temp`, and `to_xarray`

These internal utilities validate a minimum of three unique depths, orient arrays as time × depth, and attach named coordinates.

In [ ]:
formatted = fn.format_Temp(depth, temperature)
data_array, normalized_depth = fn.to_xarray(temperature, depth)
fn.control(data_array, normalized_depth), formatted.shape, data_array

## 10. Smoothing helpers

`smooth_1D` handles one vector. `smooth_temp` operates along the named depth axis and therefore also supports several profiles.

In [ ]:
noisy = temperature + np.array([0, 0.1, -0.2, 0.2, -0.1, 0.1, -0.2, 0])
smoothed_vector = fn.smooth_1D(noisy, {"window_size": 5, "order": 2})
smoothed_array = fn.smooth_temp(data_array, depth, {"window_size": 5, "order": 2})
smoothed_vector, smoothed_array

## 11. `weighted_method` and `find_peak_index`

`weighted_method` refines a density-gradient interval. `find_peak_index` returns the deepest peak above a threshold or a supplied fallback. They are implementation helpers used by the thermocline functions.

In [ ]:
density = pylake.water_density(data_array, 0.2)
gradient = density.diff("depth") / density.depth.diff("depth")
interval_index = gradient.argmax("depth")
refined = fn.weighted_method(depth, density, interval_index)
peak = fn.find_peak_index([0, 0.3, 0, 0.7, 0], 0.1, 0)
refined, peak

## 12. Remaining array and bathymetry helpers

`find_nearest_index` and `find_nearest` locate sensors. `set_nan` transfers a missing-value mask. `round_up_to_odd` produces valid smoothing windows. `check_bathy` aligns the deepest temperature and bathymetry bounds.

In [ ]:
nearest_index = fn.find_nearest_index(depth, 3.6)
nearest_depth = fn.find_nearest(depth, 3.6)
masked = fn.set_nan(np.array([1, np.nan, 3]), np.array([10.0, 20.0, 30.0]))
odd_window = fn.round_up_to_odd(6)
checked = fn.check_bathy(temperature.reshape(1, -1), bthA, bthD, depth)
nearest_index, nearest_depth, masked, odd_window, [np.shape(value) for value in checked]

## 13. Important edge cases

- Fewer than three measurements: warning and `NaN`.
- Repeated depths: clean with `depth_average` first.
- Uniform profile: no significant thermocline when the temperature range is below `mixed_cutoff`.
- `top > bottom`: layer functions raise `ValueError`.
- Irregular spacing: supported; weighted and midpoint estimates can differ.
- Multiple profiles: pass a 2-D time × depth array and matching timestamps.

In [ ]:
cases = {
    "three points": pylake.thermocline([20, 10, 9], [1, 2, 3]),
    "uniform": pylake.thermocline([10, 10, 10, 10], [0, 1, 2, 3]),
    "irregular": pylake.thermocline([21, 20.8, 20.4, 18, 12, 10, 9], [0, 0.4, 1.3, 2.7, 4.8, 7.5, 10]),
}
cases